In [2]:
!pip install textgrad -q

import argparse
import concurrent
from dotenv import load_dotenv
from tqdm import tqdm
import textgrad as tg
from textgrad.tasks import load_task
import numpy as np
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import torch
import openai
import json
import os
import pandas as pd
import re
import os
from transformers import Trainer, TrainingArguments
from peft import get_peft_model, PromptTuningConfig, TaskType
from transformers import DataCollatorForLanguageModeling
import numpy as np
import random
import textgrad as tg
import openai

load_dotenv(override=True)

zsh:1: command not found: pip


False

## Подготовка токенайзера и базовой модели

In [3]:
model_name = "tiiuae/falcon-rw-1b" 
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name)

KeyboardInterrupt: 

In [3]:
model.to(device=torch.device("cuda"))

FalconForCausalLM(
  (transformer): FalconModel(
    (word_embeddings): Embedding(50304, 2048)
    (h): ModuleList(
      (0-23): 24 x FalconDecoderLayer(
        (self_attention): FalconAttention(
          (query_key_value): FalconLinear(in_features=2048, out_features=6144, bias=True)
          (dense): FalconLinear(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (mlp): FalconMLP(
          (dense_h_to_4h): FalconLinear(in_features=2048, out_features=8192, bias=True)
          (act): GELUActivation()
          (dense_4h_to_h): FalconLinear(in_features=8192, out_features=2048, bias=True)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (rotary_emb): FalconRotaryEmbedding()
  )
  (lm_head): Li

In [4]:
model.device

device(type='cuda', index=0)

In [5]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)

In [6]:
set_seed(12)

## Подготовка датасета

In [7]:
slices = {
    "train":      "train[:1000]",
    "validation":      "train[1000:2000]",
    "test": "validation[:1000]",
}

dataset = load_dataset("glue", "sst2", split=slices)
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

## Предсказание с ручным подбором промпта на необученной модели

In [8]:
def zs_acc(dataset):
    labs = ["negative","positive"]
    correct= 0
    for ex in tqdm(dataset):
        labs = ["negative","positive"]
        inp = tokenizer(f"{ex['sentence']}\nSentiment of review (negative or positive): ", return_tensors="pt").to(model.device)
        logits = model(**inp).logits[0,-1]
        pred = labs[logits[[tokenizer(l,add_special_tokens=False).input_ids[0] for l in labs]].argmax()]
        if pred==labs[ex["label"]]: correct+=1
    return correct/len(dataset)

print("Zero‑shot acc:", zs_acc(test_data))

100%|██████████| 872/872 [00:25<00:00, 33.76it/s]

Zero‑shot acc: 0.5126146788990825


In [ ]:
os.environ["NCCL_DEBUG"] = "INFO" 
os.environ["WANDB_DISABLED"] = "true" 

## Обертка, позволяющая делать prompt-tuning v2

In [46]:


peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init="TEXT",
    num_virtual_tokens=20,
    tokenizer_name_or_path=model_name,
    prompt_tuning_init_text="Write the sentiment of that review (positive or negative): ",
)

peft_model = get_peft_model(model, peft_config)

### Далее мы будем использовать peft_model для P-tuning v2

In [47]:
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

## Функция для токенизации датасета

In [ ]:
def tokenize_dataset(dataset, template):
    def preprocess_function(examples):
        prompt = f"{template}{examples['sentence']}\nSentiment (positive or negative):"
        target = "positive" if examples["label"] == 1 else "negative"
    
        enc_pr  = tokenizer(prompt,  truncation=True, max_length=64, padding="max_length")
        enc_tgt = tokenizer(target, truncation=True, max_length=  4, padding="max_length")
    
        input_ids      = enc_pr["input_ids"] + enc_tgt["input_ids"]
        attention_mask = enc_pr["attention_mask"] + enc_tgt["attention_mask"]
    
        labels = [-100] * len(enc_pr["input_ids"]) + enc_tgt["input_ids"]
    
        return {
          "input_ids":      input_ids,
          "attention_mask": attention_mask,
          "labels":         labels,
        }  
    tokenized_dataset = dataset.map(preprocess_function, remove_columns=['sentence', 'idx', 'label'], batched=False)
    return tokenized_dataset



## Функция для обучения модели
- используется ```peft_model```
- используются различные адаптеры

  Адаптеры используются для того, чтобы не создавать каждый раз новый peft_model под разные промпты от textgrad, а делать это все в одной переменной. При обучении нового p-tuning просто используем другой tokenized_dataset

In [49]:

def train_new_model(peft_model, tokenized_dataset, num: int = 0):
    training_args = TrainingArguments(
        output_dir=f"./results{num}",
        eval_strategy="epoch",
        learning_rate=1e-3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_dir=f"./logs{num}",
        save_strategy="epoch",
        report_to="none"
    )
    adapter_name = f"adapter_{num}" if num > 0 else 'default'
    peft_config = PromptTuningConfig(
        task_type=TaskType.CAUSAL_LM,
        prompt_tuning_init="TEXT",
        num_virtual_tokens=20,
        tokenizer_name_or_path=model_name,
        prompt_tuning_init_text="Write the sentiment of that review (positive or negative): ",
    )
    peft_model.add_adapter(adapter_name, peft_config)
    peft_model.set_adapter(adapter_name)
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(1000)),
        eval_dataset=tokenized_dataset["validation"].shuffle(seed=42).select(range(200)),
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    trainer.train()
    return adapter_name, trainer

In [50]:
def get_answer(tokenizer, prompt, peft_model, max_new_tokens = 4):
    inputs = tokenizer(prompt, return_tensors="pt").to(peft_model.device)
    with torch.no_grad():
        outputs = peft_model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    try:    
        prediction = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip().split()[0].lower()
    except:
        prediction = ''
    return prediction

def get_prompt(template, sentence):
    return template + sentence

def evaluate_accuracy(peft_model, tokenizer, dataset, template, max_new_tokens=5):
    peft_model.eval()
    correct = 0
    total = 0
    
    for example in dataset:
        prompt = get_prompt(template, example['sentence'] + '\nSentiment (positive or negative):')
        prediction = get_answer(tokenizer, prompt, peft_model, max_new_tokens)
        true_label = "positive" if example["label"] == 1 else "negative"
        if prediction.startswith(true_label):
            correct += 1
        total += 1

    accuracy = correct / total
    return accuracy

In [51]:
STARTING_SYSTEM_PROMPT = """You need to write the sentiment of that review (positive or negative). Review: """


In [52]:
evaluate_accuracy(peft_model, tokenizer, test_data, STARTING_SYSTEM_PROMPT, max_new_tokens=4)

0.0

In [79]:

tokenized_dataset = tokenize_dataset(dataset, STARTING_SYSTEM_PROMPT)
adapter_name, trainer = train_new_model(peft_model, tokenized_dataset)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<ipython-input-49-9817cb749cc7>:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,1.238215
2,No log,1.216367
3,No log,1.211139


In [80]:
acc = evaluate_accuracy(peft_model, tokenizer, test_data, STARTING_SYSTEM_PROMPT, max_new_tokens=4)
print(f"Validation Accuracy: {acc:.2%}")

/usr/local/lib/python3.10/dist-packages/peft/peft_model.py:1889: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


Validation Accuracy: 84.29%


In [81]:
print("Train/Val/Test Set Lengths: ", len(train_data), len(val_data), len(test_data))

Train/Val/Test Set Lengths:  1000 1000 872


In [82]:
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("openaikey")
os.environ["OPENAI_API_KEY"] = secret_value_0
openai.api_key = os.environ["OPENAI_API_KEY"]

llm_api_eval = tg.get_engine(engine_name="gpt-3.5-turbo-0125")
tg.set_backward_engine("gpt-3.5-turbo-0125", override=True)

In [ ]:

system_prompt = tg.Variable(STARTING_SYSTEM_PROMPT, 
                            requires_grad=True,
                            role_description="structured system prompt to a somewhat capable language model that specifies the behavior and strategies for the QA task")


optimizer = tg.TGD(parameters=[system_prompt])

In [84]:
eval_engine = tg.get_engine("gpt-3.5-turbo-0125")

_, _, _, eval_fn = load_task("BBH_object_counting", eval_engine)

In [ ]:
from types import MethodType
from typing import Dict
from textgrad.variable import Variable
from textgrad.autograd.llm_ops import LLMCall, BackwardContext


def echo_forward(self, inputs: Dict[str, tg.variable.Variable],
    response_role_description: str = None,):

    if response_role_description is None:
            response_role_description = f"Output of the string-based function with purpose: {self.function_purpose}"
    response_string = str(int(inputs['prediction'].value == inputs['ground_truth_answer'].value))
    response = Variable(
        value=response_string,
        predecessors=list(inputs.values()),
        role_description=response_role_description
    )
    
    
    response.set_grad_fn(BackwardContext(backward_fn=self.backward, 
                                         response=response, 
                                         function_purpose=self.function_purpose,
                                         inputs=inputs))

    return response
eval_fn.forward = MethodType(echo_forward, eval_fn)

In [87]:
p = tg.Variable('positive', requires_grad=False, role_description="true answer")
pp = tg.Variable('positive', requires_grad=False, role_description="predicted answer")


eval_fn(inputs=dict(prediction=p, ground_truth_answer=pp))

Variable(value=1, role=Output of the string-based function with purpose: The runtime of string-based function that checks if the prediction is correct., grads=set())

In [88]:
emotions = ['negative', 'positive']

def eval_sample(item, eval_fn, peft_model):
    """
    This function allows us to evaluate if an answer to a question in the prompt is a good answer.

    """
    y, x = item
    y = str(y)
    x = str(x)
    x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
    y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
    prompt = f"{system_prompt.value}{x.value}\nSentiment (positive or negative):"
    prediction = tg.Variable(get_answer(tokenizer, prompt, peft_model), requires_grad=False, role_description="prediction from a model")
    return int(prediction.value == y.value)

def eval_dataset(test_set, eval_fn, peft_model, max_samples: int=None):
    if max_samples is None:
        max_samples = len(test_set['sentence'])
    accuracy_list = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        futures = []
        for i in range(len(test_set['sentence'])):
            future = executor.submit(eval_sample, (emotions[test_set['label'][i]], test_set['sentence'][i]), eval_fn, peft_model)
            futures.append(future)
            if len(futures) >= max_samples:
                break
        tqdm_loader = tqdm(concurrent.futures.as_completed(futures), total=len(futures), position=0)
        for future in tqdm_loader:
            acc_item = future.result()
            try:
                pupupu = float(acc_item)
            except:
                pupupu = 0.
            accuracy_list.append(pupupu)
            tqdm_loader.set_description(f"Accuracy: {np.mean(accuracy_list)}")
    return accuracy_list 

def run_validation_revert(system_prompt: tg.Variable, results, peft_model, eval_fn, val_data, i = 1):
    tokenized_dataset = tokenize_dataset(dataset, system_prompt.value)
    adapter_name, trainer = train_new_model(peft_model, tokenized_dataset, i)
    val_performance = np.mean(eval_dataset(val_data, eval_fn, peft_model))
    previous_performance = np.mean(results["validation_acc"][-1])
    print("val_performance: ", val_performance)
    print("previous_performance: ", previous_performance)
    previous_prompt = results["prompt"][-1]
    
    if val_performance < previous_performance:
        print(f"rejected prompt: {system_prompt.value}")
        system_prompt.set_value(previous_prompt)
        val_performance = previous_performance

    return adapter_name, trainer

In [89]:
eval_dataset(test_data, eval_fn, peft_model, max_samples =2)

  0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/peft/peft_model.py:1889: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Accuracy: 0.5: 100%|██████████| 2/2 [00:00<00:00, 10.02it/s]


[1.0, 0.0]

In [90]:
eval_sample(("it 's a charming and often affecting journey . ", 
'positive'), eval_fn, peft_model)

0

In [91]:
peft_model.set_adapter("default")

In [92]:
system_prompt.value

'You need to write the sentiment of that review (positive or negative). Review: '

In [93]:


results = {"test_acc": [], "prompt": [], "validation_acc": [], 'adapter_names': [], 'trainers': []}
results["test_acc"].append(eval_dataset(test_data.shuffle(seed=42).select(range(20)), eval_fn, peft_model))
results["validation_acc"].append(eval_dataset(val_data.shuffle(seed=42).select(range(20)), eval_fn, peft_model))
results["adapter_names"].append(adapter_name)
results["prompt"].append(system_prompt.get_value())
results["trainers"].append(trainer)


Accuracy: 0.7: 100%|██████████| 20/20 [00:01<00:00, 10.84it/s]               


In [94]:
[i for i in results['prompt']]


['You need to write the sentiment of that review (positive or negative). Review: ']

In [96]:
train_loader = tg.tasks.DataLoader(train_set, batch_size=2, shuffle=True)

In [ ]:
for epoch in range(5):
    for steps, batch in enumerate((pbar := tqdm(train_loader, position=0))):
        pbar.set_description(f"Training step {steps}. Epoch {epoch}")
        losses = []
        for (x, y) in zip(batch[0], batch[1]):
            x = tg.Variable(x, requires_grad=False, role_description="query to the language model")
            y = tg.Variable(y, requires_grad=False, role_description="correct answer for the query")
            prompt = f"{system_prompt.value}{x.value}\nSentiment (positive or negative):"
            prediction = get_answer(tokenizer, prompt, peft_model)
            prediction = tg.Variable(prediction, requires_grad=False, role_description="response from a model")
            try:
                eval_output_variable = eval_fn(inputs=dict(prediction=prediction, ground_truth_answer=y))
            except:
                eval_output_variable = eval_fn([x, y, prediction])
            losses.append(eval_output_variable)
        total_loss = tg.sum(losses)
        total_loss.backward()
        try:
            optimizer.step()
        except IndexError as e:
            print(f"Optimizer step failed at epoch {epoch}, step {steps}: {e}")
            continue
        
        adapter_name, trainer = run_validation_revert(system_prompt, results, peft_model, eval_fn, val_data, epoch + 1)
        print("sys prompt: ", system_prompt)
        results["test_acc"].append(eval_dataset(test_data, eval_fn, peft_model))
        results["validation_acc"].append(eval_dataset(val_data, eval_fn, peft_model))
        results["adapter_names"].append(adapter_name)
        results["prompt"].append(system_prompt.get_value())
        results["trainers"].append(trainer)
        if steps == 1:
            break

Training step 0. Epoch 0: : 0it [00:00, ?it/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

<ipython-input-49-9817cb749cc7>:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,1.139869
2,No log,1.105639
3,No log,1.099597


  0%|          | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/peft/peft_model.py:1889: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
Accuracy: 0.853: 100%|██████████| 1000/1000 [01:32<00:00, 10.80it/s]            


val_performance:  0.853
previous_performance:  0.7
sys prompt:  Given the review text, determine and write down whether the sentiment of the review is positive or negative.


Accuracy: 0.8778195488721805:  53%|█████▎    | 530/1000 [00:48<00:44, 10.67it/s]